In [3]:
import os, requests, pandas as pd
# from config import ADZUNA_APP_ID, ADZUNA_APP_KEY
ADZUNA_APP_ID="774639e9"
ADZUNA_APP_KEY="5473959593308b776f8850763aac4880"
def run(job_title, city, country_code, data_dir):
    os.makedirs(data_dir, exist_ok=True)

    url = f"https://api.adzuna.com/v1/api/jobs/gb/search/1{country_code}/search/1"
    params = {
        "app_id": ADZUNA_APP_ID,
        "app_key": ADZUNA_APP_KEY,
        "results_per_page": 50,
        "what": job_title,
        "where": city,
        "content-type": "application/json",
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json().get("results", [])

    jobs = pd.DataFrame([{
        "title":       r.get("title"),
        "company":     r.get("company", {}).get("display_name"),
        "location":    r.get("location", {}).get("display_name"),
        "description": r.get("description"),
        "url":         r.get("redirect_url"),
        "salary_min":  r.get("salary_min"),
        "salary_max":  r.get("salary_max"),
        "date_posted": r.get("created"),
        "source":      "adzuna",
    } for r in results])

    filename = os.path.join(data_dir, "adzuna.csv")
    jobs.to_csv(filename, index=False)
    print(f"[Adzuna] Found {len(jobs)} jobs → {filename}")